In [1]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import os
import re
import time
import matplotlib
import sys
import pandas as pd

sys.path.append('../')
from carpet_reconstruction import GetCarpetSignal, VEM, get_rho, NKG, num_sort, get_xy_fits

from matplotlib.backends.backend_pdf import PdfPages
from skimage.feature import peak_local_max
from scipy.optimize import minimize
from scipy.optimize import curve_fit
from scipy.special import gamma

In [2]:
# matplotlib.use('pdf') #Чтобы не убить оперативку
plt.rcParams['figure.dpi'] = 400
# plt.style.use('fivethirtyeight')

In [3]:
path = '../Data/proton_data/'

filenames = [path + filename for filename in os.listdir(path)]
filenames.sort(key=num_sort)

In [4]:
high_energy_files = filenames[28:]
high_energy_files

['../Data/proton_data/output28.root']

In [5]:
pdf = PdfPages("NKG_test.pdf")

fig, axs = plt.subplots(1, 2, figsize=(16, 9))

for filename in high_energy_files:

    print(filename)
    
    file = uproot.open(filename)

    names = np.array(file.keys())
    times = names[:len(names) // 3]
    cover_hists = names[len(names) // 3::2]

    cover_signals = np.zeros((len(cover_hists), 20, 20))

    for i, hist in enumerate(cover_hists):
        cov_hist = np.copy(np.flip(file[hist].values().T, 0))
        cover_signals[i] = VEM(GetCarpetSignal(cov_hist))
    
    time_array = np.zeros((len(times), 4))
    
    for i, t in enumerate(times):
        time_array[i] = file[t]['t_ns'].array(library='np')[1:5]

    for i in range(len(cover_signals)):

        sig = cover_signals[i]
        times = time_array[i]

        if ((np.all(times)) and (np.count_nonzero(sig) > 50)):

            r, rho = get_rho(sig, times)
            
            axs[0].scatter(r, rho, label='Прямые вычисления')
        
            try:
                popt, pcov = curve_fit(NKG, r, rho, bounds=([0.4, 10], [1.8, 10**8]))
                axs[0].plot(r, NKG(r, *popt), '-ro', linewidth = 2, markersize=3, label='NKG fit: s=%5.3f, Ne=%5.3f' % tuple(popt))
                axs[0].set_xlabel('r в плоскости ливня, м', fontsize=10)
                axs[0].set_ylabel('ρ, ч/м$^2$', fontsize=10)
        
                axs[0].legend(fontsize=6)
            except RuntimeError:
                ...
                # print('Unable to fit')
            except TypeError:
                ...
                # print('Unable to fit again')
            except ValueError:
                ...
                # print('Data empty')
    
            fig.suptitle(cover_hists[i], fontsize=10)
    
            axs[1].imshow(sig, cmap='turbo')

            row0, col0 = get_xy_fits(sig)
            plt.plot(col0, row0, 'mo')
    
            for j in range(sig.shape[0]):
                for k in range(sig.shape[1]):
                    text = axs[1].text(k, j, round(sig[j, k]), ha="center", va="center", color="w", fontsize=5)
            
            axs[1].set_xticks([])
            axs[1].set_yticks([])
            
            plt.tight_layout()
        
            pdf.savefig(fig)

            for ax in axs:
                ax.clear()            

pdf.close()
print('fin')

../Data/proton_data/output28.root


KeyboardInterrupt: 

Error in callback <function flush_figures at 0x0000019949AABD30> (for post_execute):


KeyboardInterrupt: 